<a href="https://colab.research.google.com/github/Roberto-Raposo/TCC_Treinamento_Garbage_Classification/blob/main/Treinamento_Garbage_Classification_ResNet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importação das bibliotecas

import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

from google.colab import drive

In [ ]:
# Configurações do projeto

BATCH_SIZE = 16
IMAGE_SIZE = (224, 224)
SEED = 42
NUM_CLASSES = 10
EPOCHS = 20
FINE_TUNE_EPOCHS = 15
FINE_TUNE_LR = 1e-5
UNFREEZE_FRACTION = 0.3

In [ ]:
# Reprodutibilidade

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Montagem do Google Drive

drive.mount('/content/drive')

In [ ]:
# Estrutura de diretórios (mesma do notebook do EfficientNetV2S)

PROJECT_PATH = "/content/drive/MyDrive/TCC"

folders = ["dataset", "models", "results", "graphs", "history", "notebooks"]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

print("Estrutura criada com sucesso!")

In [ ]:
# Extracao do Dataset

import zipfile

zip_path = "/content/drive/MyDrive/garbage-classification-v2.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extraído com sucesso!")

In [ ]:
# Definicao do Dataset

DATASET_PATH = "/content/dataset/standardized_384"

classes = sorted(os.listdir(DATASET_PATH))

print("Número de classes:", len(classes))
print(classes)

Bloco 2 — Coleta, split estratificado e class weighting

In [ ]:
# Coleta de caminhos e rotulos de todas as imagens

file_paths = []
labels = []

for idx, classe in enumerate(classes):
    pasta = os.path.join(DATASET_PATH, classe)
    for arquivo in os.listdir(pasta):
        file_paths.append(os.path.join(pasta, arquivo))
        labels.append(idx)

file_paths = np.array(file_paths)
labels = np.array(labels)

print("Total de imagens:", len(file_paths))

In [ ]:
# Divisao Estratificada em Treino, Validacao e Teste (80% / 10% / 10%)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    file_paths, labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=0.5,
    stratify=temp_labels,
    random_state=SEED
)

print("Treino:", len(train_paths))
print("Validação:", len(val_paths))
print("Teste:", len(test_paths))

In [ ]:
# Calculo dos pesos de classe (class weighting)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weight_dict = dict(enumerate(class_weights_array))

print("Pesos por classe:")
for idx, peso in class_weight_dict.items():
    print(f"{classes[idx]}: {peso:.3f}")

Bloco 3 — Pipeline tf.data (224×224, específico do ResNet50)

In [ ]:
# Funcao de carregamento e criacao dos datasets

AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE)
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

def make_dataset(paths, labels, shuffle=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_dataset = make_dataset(train_paths, train_labels, shuffle=True)
validation_dataset = make_dataset(val_paths, val_labels, shuffle=False)
test_dataset = make_dataset(test_paths, test_labels, shuffle=False)

In [ ]:
# Classes Carregadas

class_names = classes
print(class_names)

In [ ]:
# Formato Dos Dados

for images, labels_batch in train_dataset.take(1):
    print("Formato das imagens:", images.shape)
    print("Formato dos labels:", labels_batch.shape)

In [ ]:
# Configuração do Data Augmentation

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1)
])

Bloco 4 — Modelo (Etapa 1: base congelada)

In [ ]:
# Carregamento do ResNet50

base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

In [ ]:
# Congelamento das camadas

base_model.trainable = False

In [ ]:
# Construcao da Arquitetura

inputs = keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)          # só age durante o treino (training=True)
x = layers.Lambda(resnet_preprocess_input)(x)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [ ]:
# Resumo da Arquitetura

model.summary()

In [ ]:
# Compilacao

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

Bloco 5 — Callbacks e treino (Etapa 1)

In [ ]:
# Callbacks (Etapa 1)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(PROJECT_PATH, "models", "resnet50_best.keras"),
    monitor="val_loss", save_best_only=True, verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6, verbose=1
)

callbacks = [early_stopping, checkpoint, reduce_lr]

In [ ]:
# Treinamento - Etapa 1 (cabeça de classificação)

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# Salvamento do historico (Etapa 1)

history_df = pd.DataFrame(history.history)

history_df.to_csv(
    os.path.join(PROJECT_PATH, "history", "history_resnet_transfer_learning.csv"),
    index=False
)

print("Histórico salvo com sucesso!")

Bloco 6 — Fine-tuning (Etapa 2)

In [ ]:
# Descongelamento parcial do modelo base (Etapa 2 - Fine-Tuning)

base_model.trainable = True

total_layers = len(base_model.layers)
fine_tune_at = int(total_layers * (1 - UNFREEZE_FRACTION))

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Total de camadas no modelo base: {total_layers}")
print(f"Camadas congeladas: {fine_tune_at}")
print(f"Camadas descongeladas (treináveis): {total_layers - fine_tune_at}")

In [ ]:
# Recompilacao do modelo para Fine-Tuning

fine_tune_optimizer = tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR)

model.compile(
    optimizer=fine_tune_optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Modelo recompilado com learning rate:", FINE_TUNE_LR)

In [ ]:
# Callbacks para o Fine-Tuning

early_stopping_ft = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
)

checkpoint_ft = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(PROJECT_PATH, "models", "resnet50_finetuned_best.keras"),
    monitor="val_loss", save_best_only=True, verbose=1
)

reduce_lr_ft = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1
)

callbacks_ft = [early_stopping_ft, checkpoint_ft, reduce_lr_ft]

In [ ]:
# Treinamento - Etapa 2 (Fine-Tuning)

history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks_ft,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# Salvamento do historico do Fine-Tuning

history_finetune_df = pd.DataFrame(history_finetune.history)

history_finetune_df.to_csv(
    os.path.join(PROJECT_PATH, "history", "history_resnet_finetuning.csv"),
    index=False
)

print("Histórico do Fine-Tuning salvo com sucesso!")

Bloco 7 — Avaliação

In [ ]:
# Avaliacao do Conjunto de teste

test_loss, test_accuracy = model.evaluate(test_dataset, verbose=1)

print(f"Loss no conjunto de teste: {test_loss:.4f}")
print(f"Acurácia no conjunto de teste: {test_accuracy:.4f}")

In [ ]:
# Geracao das Predicoes

y_true = []
y_pred = []

for images, labels_batch in test_dataset:
    predictions = model(images, training=False)
    true_classes = tf.argmax(labels_batch, axis=1)
    predicted_classes = tf.argmax(predictions, axis=1)
    y_true.extend(true_classes.numpy().tolist())
    y_pred.extend(predicted_classes.numpy().tolist())

In [ ]:
# Matriz de confusao

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Matriz de Confusão - ResNet50")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_PATH, "graphs", "04_matriz_confusao_resnet50.png"),
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Classification Report

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Salvamento do Relatorio

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    os.path.join(PROJECT_PATH, "results", "classification_report_resnet50.csv")
)

report_df